# EEL891 - 2026.01 -  Trabalho 1 - Aprendizado de máquina - Thiago Moutinho de Carvalho Maksoud - DRE 119048139

Imports

In [285]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

Configurações para os gráficos

In [286]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### Alocando os dataframes de teste e treino

In [ ]:
train_df = pd.read_csv('./data/trabalho1/conjunto_de_treinamento.csv')
test_df = pd.read_csv('./data/trabalho1/conjunto_de_teste.csv')

#### Analisando o dataframe vendo as 5 primeiras linhas

In [288]:
train_df.head()

,id_solicitante,produto_solicitado,dia_vencimento,forma_envio_solicitacao,tipo_endereco,sexo,idade,estado_civil,qtde_dependentes,grau_instrucao,nacionalidade,estado_onde_nasceu,estado_onde_reside,possui_telefone_residencial,codigo_area_telefone_residencial,tipo_residencia,meses_na_residencia,possui_telefone_celular,possui_email,renda_mensal_regular,renda_extra,possui_cartao_visa,possui_cartao_mastercard,possui_cartao_diners,possui_cartao_amex,possui_outros_cartoes,qtde_contas_bancarias,qtde_contas_bancarias_especiais,valor_patrimonio_pessoal,possui_carro,vinculo_formal_com_empresa,estado_onde_trabalha,possui_telefone_trabalho,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente
0,1,1,10,presencial,1,M,85,2,0,0,1,CE,CE,Y,107,1.0,12.0,N,0,480.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,N,,0,9.0,1.0,0.0,0.0,600.0,600.0,0
1,2,1,25,internet,1,F,38,1,0,0,1,SE,SE,Y,91,1.0,5.0,N,1,380.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,2.0,5.0,NaN,NaN,492.0,492.0,0
2,3,1,20,internet,1,F,37,2,0,0,1,BA,BA,Y,90,5.0,1.0,N,1,600.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,NaN,NaN,NaN,NaN,450.0,450.0,1
3,4,1,20,internet,1,M,37,1,1,0,1,RS,RS,Y,54,1.0,1.0,N,1,460.0,0.0,0,0,0,0,0,0,0,0.0,0,Y,RS,Y,54,0,9.0,2.0,NaN,NaN,932.0,932.0,1
4,5,7,1,internet,1,F,51,1,3,0,1,BA,BA,Y,86,0.0,1.0,N,1,687.0,600.0,0,0,0,0,0,0,0,0.0,1,Y,BA,N,,0,9.0,5.0,NaN,NaN,440.0,440.0,1


# Pré processamento - limpar e normalizar o dataframe

In [289]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 42 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id_solicitante                    20000 non-null  int64  
 1   produto_solicitado                20000 non-null  int64  
 2   dia_vencimento                    20000 non-null  int64  
 3   forma_envio_solicitacao           20000 non-null  str    
 4   tipo_endereco                     20000 non-null  int64  
 5   sexo                              20000 non-null  str    
 6   idade                             20000 non-null  int64  
 7   estado_civil                      20000 non-null  int64  
 8   qtde_dependentes                  20000 non-null  int64  
 9   grau_instrucao                    20000 non-null  int64  
 10  nacionalidade                     20000 non-null  int64  
 11  estado_onde_nasceu                20000 non-null  str    
 12  estado_onde_res

## Temos 2 tarefas principais, transformar dados string em números, e preencher entradas vazias

### Prenchendo entradas vazias

In [290]:
nulos = train_df.isna().sum()

# Filtra e exibe apenas as colunas que possuem mais de 0 nulos
colunas_com_nulos = nulos[nulos > 0]
print("--- Colunas com valores NaN ---")
print(colunas_com_nulos)

--- Colunas com valores NaN ---
tipo_residencia                 536
meses_na_residencia            1450
profissao                      3097
ocupacao                       2978
profissao_companheiro         11514
grau_instrucao_companheiro    12860
dtype: int64


# 1. Preenchendo as categorias estruturais e profissionais com -1

In [291]:

colunas_para_placeholder = [
    'profissao_companheiro', 
    'grau_instrucao_companheiro', 
    'profissao', 
    'ocupacao', 
    'tipo_residencia'
]
train_df[colunas_para_placeholder] = train_df[colunas_para_placeholder].fillna(-1)
test_df[colunas_para_placeholder] = test_df[colunas_para_placeholder].fillna(-1)


# 2. Preenchendo os meses de residência com a mediana

In [292]:
mediana_meses = train_df['meses_na_residencia'].median()
train_df['meses_na_residencia'] = train_df['meses_na_residencia'].fillna(mediana_meses)
test_df['meses_na_residencia'] = test_df['meses_na_residencia'].fillna(mediana_meses)


In [293]:
train_df.head()

,id_solicitante,produto_solicitado,dia_vencimento,forma_envio_solicitacao,tipo_endereco,sexo,idade,estado_civil,qtde_dependentes,grau_instrucao,nacionalidade,estado_onde_nasceu,estado_onde_reside,possui_telefone_residencial,codigo_area_telefone_residencial,tipo_residencia,meses_na_residencia,possui_telefone_celular,possui_email,renda_mensal_regular,renda_extra,possui_cartao_visa,possui_cartao_mastercard,possui_cartao_diners,possui_cartao_amex,possui_outros_cartoes,qtde_contas_bancarias,qtde_contas_bancarias_especiais,valor_patrimonio_pessoal,possui_carro,vinculo_formal_com_empresa,estado_onde_trabalha,possui_telefone_trabalho,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente
0,1,1,10,presencial,1,M,85,2,0,0,1,CE,CE,Y,107,1.0,12.0,N,0,480.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,N,,0,9.0,1.0,0.0,0.0,600.0,600.0,0
1,2,1,25,internet,1,F,38,1,0,0,1,SE,SE,Y,91,1.0,5.0,N,1,380.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,2.0,5.0,-1.0,-1.0,492.0,492.0,0
2,3,1,20,internet,1,F,37,2,0,0,1,BA,BA,Y,90,5.0,1.0,N,1,600.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,-1.0,-1.0,-1.0,-1.0,450.0,450.0,1
3,4,1,20,internet,1,M,37,1,1,0,1,RS,RS,Y,54,1.0,1.0,N,1,460.0,0.0,0,0,0,0,0,0,0,0.0,0,Y,RS,Y,54,0,9.0,2.0,-1.0,-1.0,932.0,932.0,1
4,5,7,1,internet,1,F,51,1,3,0,1,BA,BA,Y,86,0.0,1.0,N,1,687.0,600.0,0,0,0,0,0,0,0,0.0,1,Y,BA,N,,0,9.0,5.0,-1.0,-1.0,440.0,440.0,1


Checando a coluna codigo_area_telefone_trabalho

In [294]:
train_df['codigo_area_telefone_trabalho']

0          
1          
2          
3        54
4          
         ..
19995      
19996      
19997      
19998      
19999      
Name: codigo_area_telefone_trabalho, Length: 20000, dtype: str

Muitos valores vazios

In [295]:
train_df.head(10)

,id_solicitante,produto_solicitado,dia_vencimento,forma_envio_solicitacao,tipo_endereco,sexo,idade,estado_civil,qtde_dependentes,grau_instrucao,nacionalidade,estado_onde_nasceu,estado_onde_reside,possui_telefone_residencial,codigo_area_telefone_residencial,tipo_residencia,meses_na_residencia,possui_telefone_celular,possui_email,renda_mensal_regular,renda_extra,possui_cartao_visa,possui_cartao_mastercard,possui_cartao_diners,possui_cartao_amex,possui_outros_cartoes,qtde_contas_bancarias,qtde_contas_bancarias_especiais,valor_patrimonio_pessoal,possui_carro,vinculo_formal_com_empresa,estado_onde_trabalha,possui_telefone_trabalho,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente
0,1,1,10,presencial,1,M,85,2,0,0,1,CE,CE,Y,107,1.0,12.0,N,0,480.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,N,,0,9.0,1.0,0.0,0.0,600.0,600.0,0
1,2,1,25,internet,1,F,38,1,0,0,1,SE,SE,Y,91,1.0,5.0,N,1,380.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,2.0,5.0,-1.0,-1.0,492.0,492.0,0
2,3,1,20,internet,1,F,37,2,0,0,1,BA,BA,Y,90,5.0,1.0,N,1,600.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,-1.0,-1.0,-1.0,-1.0,450.0,450.0,1
3,4,1,20,internet,1,M,37,1,1,0,1,RS,RS,Y,54,1.0,1.0,N,1,460.0,0.0,0,0,0,0,0,0,0,0.0,0,Y,RS,Y,54,0,9.0,2.0,-1.0,-1.0,932.0,932.0,1
4,5,7,1,internet,1,F,51,1,3,0,1,BA,BA,Y,86,0.0,1.0,N,1,687.0,600.0,0,0,0,0,0,0,0,0.0,1,Y,BA,N,,0,9.0,5.0,-1.0,-1.0,440.0,440.0,1
5,6,1,20,presencial,1,M,21,1,1,0,1,CE,CE,Y,107,5.0,2.0,N,0,382.0,0.0,1,0,0,0,0,0,0,0.0,1,Y,CE,Y,107,0,9.0,2.0,0.0,0.0,628.0,628.0,1
6,7,1,15,presencial,1,F,64,4,2,0,1,SP,SP,Y,16,1.0,0.0,N,1,350.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,N,,0,10.0,1.0,0.0,0.0,190.0,190.0,1
7,8,1,5,internet,1,F,20,1,0,0,1,ES,ES,Y,25,1.0,5.0,N,1,800.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,N,,0,-1.0,-1.0,-1.0,-1.0,299.0,299.0,1
8,9,2,25,internet,1,F,39,2,2,0,1,GO,GO,Y,67,1.0,3.0,N,1,1200.0,0.0,1,0,0,0,0,0,0,0.0,0,Y,,Y,69,0,9.0,2.0,9.0,4.0,756.0,756.0,0
9,10,1,10,presencial,1,M,44,2,2,0,1,RS,RS,N,,1.0,15.0,N,0,749.0,0.0,0,0,0,0,0,1,1,0.0,1,Y,RS,N,,0,9.0,2.0,16.0,4.0,960.0,960.0,1


In [296]:
train_df = pd.get_dummies(train_df, columns=['forma_envio_solicitacao', 'sexo', 'possui_telefone_trabalho' ], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['forma_envio_solicitacao', 'sexo', 'possui_telefone_trabalho' ], drop_first=True)

In [297]:
pd.set_option('display.max_columns', None)
train_df.head()

,id_solicitante,produto_solicitado,dia_vencimento,tipo_endereco,idade,estado_civil,qtde_dependentes,grau_instrucao,nacionalidade,estado_onde_nasceu,estado_onde_reside,possui_telefone_residencial,codigo_area_telefone_residencial,tipo_residencia,meses_na_residencia,possui_telefone_celular,possui_email,renda_mensal_regular,renda_extra,possui_cartao_visa,possui_cartao_mastercard,possui_cartao_diners,possui_cartao_amex,possui_outros_cartoes,qtde_contas_bancarias,qtde_contas_bancarias_especiais,valor_patrimonio_pessoal,possui_carro,vinculo_formal_com_empresa,estado_onde_trabalha,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente,forma_envio_solicitacao_internet,forma_envio_solicitacao_presencial,sexo_F,sexo_M,sexo_N,possui_telefone_trabalho_Y
0,1,1,10,1,85,2,0,0,1,CE,CE,Y,107,1.0,12.0,N,0,480.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,,0,9.0,1.0,0.0,0.0,600.0,600.0,0,False,True,False,True,False,False
1,2,1,25,1,38,1,0,0,1,SE,SE,Y,91,1.0,5.0,N,1,380.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,2.0,5.0,-1.0,-1.0,492.0,492.0,0,True,False,True,False,False,False
2,3,1,20,1,37,2,0,0,1,BA,BA,Y,90,5.0,1.0,N,1,600.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,-1.0,-1.0,-1.0,-1.0,450.0,450.0,1,True,False,True,False,False,False
3,4,1,20,1,37,1,1,0,1,RS,RS,Y,54,1.0,1.0,N,1,460.0,0.0,0,0,0,0,0,0,0,0.0,0,Y,RS,54,0,9.0,2.0,-1.0,-1.0,932.0,932.0,1,True,False,False,True,False,True
4,5,7,1,1,51,1,3,0,1,BA,BA,Y,86,0.0,1.0,N,1,687.0,600.0,0,0,0,0,0,0,0,0.0,1,Y,BA,,0,9.0,5.0,-1.0,-1.0,440.0,440.0,1,True,False,True,False,False,False


In [298]:
train_df.head(10)

,id_solicitante,produto_solicitado,dia_vencimento,tipo_endereco,idade,estado_civil,qtde_dependentes,grau_instrucao,nacionalidade,estado_onde_nasceu,estado_onde_reside,possui_telefone_residencial,codigo_area_telefone_residencial,tipo_residencia,meses_na_residencia,possui_telefone_celular,possui_email,renda_mensal_regular,renda_extra,possui_cartao_visa,possui_cartao_mastercard,possui_cartao_diners,possui_cartao_amex,possui_outros_cartoes,qtde_contas_bancarias,qtde_contas_bancarias_especiais,valor_patrimonio_pessoal,possui_carro,vinculo_formal_com_empresa,estado_onde_trabalha,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente,forma_envio_solicitacao_internet,forma_envio_solicitacao_presencial,sexo_F,sexo_M,sexo_N,possui_telefone_trabalho_Y
0,1,1,10,1,85,2,0,0,1,CE,CE,Y,107,1.0,12.0,N,0,480.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,,0,9.0,1.0,0.0,0.0,600.0,600.0,0,False,True,False,True,False,False
1,2,1,25,1,38,1,0,0,1,SE,SE,Y,91,1.0,5.0,N,1,380.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,2.0,5.0,-1.0,-1.0,492.0,492.0,0,True,False,True,False,False,False
2,3,1,20,1,37,2,0,0,1,BA,BA,Y,90,5.0,1.0,N,1,600.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,-1.0,-1.0,-1.0,-1.0,450.0,450.0,1,True,False,True,False,False,False
3,4,1,20,1,37,1,1,0,1,RS,RS,Y,54,1.0,1.0,N,1,460.0,0.0,0,0,0,0,0,0,0,0.0,0,Y,RS,54,0,9.0,2.0,-1.0,-1.0,932.0,932.0,1,True,False,False,True,False,True
4,5,7,1,1,51,1,3,0,1,BA,BA,Y,86,0.0,1.0,N,1,687.0,600.0,0,0,0,0,0,0,0,0.0,1,Y,BA,,0,9.0,5.0,-1.0,-1.0,440.0,440.0,1,True,False,True,False,False,False
5,6,1,20,1,21,1,1,0,1,CE,CE,Y,107,5.0,2.0,N,0,382.0,0.0,1,0,0,0,0,0,0,0.0,1,Y,CE,107,0,9.0,2.0,0.0,0.0,628.0,628.0,1,False,True,False,True,False,True
6,7,1,15,1,64,4,2,0,1,SP,SP,Y,16,1.0,0.0,N,1,350.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,,0,10.0,1.0,0.0,0.0,190.0,190.0,1,False,True,True,False,False,False
7,8,1,5,1,20,1,0,0,1,ES,ES,Y,25,1.0,5.0,N,1,800.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,-1.0,-1.0,-1.0,-1.0,299.0,299.0,1,True,False,True,False,False,False
8,9,2,25,1,39,2,2,0,1,GO,GO,Y,67,1.0,3.0,N,1,1200.0,0.0,1,0,0,0,0,0,0,0.0,0,Y,,69,0,9.0,2.0,9.0,4.0,756.0,756.0,0,True,False,True,False,False,True
9,10,1,10,1,44,2,2,0,1,RS,RS,N,,1.0,15.0,N,0,749.0,0.0,0,0,0,0,0,1,1,0.0,1,Y,RS,,0,9.0,2.0,16.0,4.0,960.0,960.0,1,False,True,False,True,False,False


In [299]:
train_df['codigo_area_telefone_trabalho'] = train_df['codigo_area_telefone_trabalho'].fillna(-1)
test_df['codigo_area_telefone_trabalho'] = test_df['codigo_area_telefone_trabalho'].fillna(-1)
train_df['codigo_area_telefone_trabalho'] = train_df['codigo_area_telefone_trabalho'].astype(str)
test_df['codigo_area_telefone_trabalho'] = test_df['codigo_area_telefone_trabalho'].astype(str)

In [300]:
train_df.head(10)

,id_solicitante,produto_solicitado,dia_vencimento,tipo_endereco,idade,estado_civil,qtde_dependentes,grau_instrucao,nacionalidade,estado_onde_nasceu,estado_onde_reside,possui_telefone_residencial,codigo_area_telefone_residencial,tipo_residencia,meses_na_residencia,possui_telefone_celular,possui_email,renda_mensal_regular,renda_extra,possui_cartao_visa,possui_cartao_mastercard,possui_cartao_diners,possui_cartao_amex,possui_outros_cartoes,qtde_contas_bancarias,qtde_contas_bancarias_especiais,valor_patrimonio_pessoal,possui_carro,vinculo_formal_com_empresa,estado_onde_trabalha,codigo_area_telefone_trabalho,meses_no_trabalho,profissao,ocupacao,profissao_companheiro,grau_instrucao_companheiro,local_onde_reside,local_onde_trabalha,inadimplente,forma_envio_solicitacao_internet,forma_envio_solicitacao_presencial,sexo_F,sexo_M,sexo_N,possui_telefone_trabalho_Y
0,1,1,10,1,85,2,0,0,1,CE,CE,Y,107,1.0,12.0,N,0,480.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,,0,9.0,1.0,0.0,0.0,600.0,600.0,0,False,True,False,True,False,False
1,2,1,25,1,38,1,0,0,1,SE,SE,Y,91,1.0,5.0,N,1,380.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,2.0,5.0,-1.0,-1.0,492.0,492.0,0,True,False,True,False,False,False
2,3,1,20,1,37,2,0,0,1,BA,BA,Y,90,5.0,1.0,N,1,600.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,-1.0,-1.0,-1.0,-1.0,450.0,450.0,1,True,False,True,False,False,False
3,4,1,20,1,37,1,1,0,1,RS,RS,Y,54,1.0,1.0,N,1,460.0,0.0,0,0,0,0,0,0,0,0.0,0,Y,RS,54,0,9.0,2.0,-1.0,-1.0,932.0,932.0,1,True,False,False,True,False,True
4,5,7,1,1,51,1,3,0,1,BA,BA,Y,86,0.0,1.0,N,1,687.0,600.0,0,0,0,0,0,0,0,0.0,1,Y,BA,,0,9.0,5.0,-1.0,-1.0,440.0,440.0,1,True,False,True,False,False,False
5,6,1,20,1,21,1,1,0,1,CE,CE,Y,107,5.0,2.0,N,0,382.0,0.0,1,0,0,0,0,0,0,0.0,1,Y,CE,107,0,9.0,2.0,0.0,0.0,628.0,628.0,1,False,True,False,True,False,True
6,7,1,15,1,64,4,2,0,1,SP,SP,Y,16,1.0,0.0,N,1,350.0,0.0,0,0,0,0,0,1,1,0.0,1,N,,,0,10.0,1.0,0.0,0.0,190.0,190.0,1,False,True,True,False,False,False
7,8,1,5,1,20,1,0,0,1,ES,ES,Y,25,1.0,5.0,N,1,800.0,0.0,0,0,0,0,0,0,0,0.0,0,N,,,0,-1.0,-1.0,-1.0,-1.0,299.0,299.0,1,True,False,True,False,False,False
8,9,2,25,1,39,2,2,0,1,GO,GO,Y,67,1.0,3.0,N,1,1200.0,0.0,1,0,0,0,0,0,0,0.0,0,Y,,69,0,9.0,2.0,9.0,4.0,756.0,756.0,0,True,False,True,False,False,True
9,10,1,10,1,44,2,2,0,1,RS,RS,N,,1.0,15.0,N,0,749.0,0.0,0,0,0,0,0,1,1,0.0,1,Y,RS,,0,9.0,2.0,16.0,4.0,960.0,960.0,1,False,True,False,True,False,False


## Estados ainda estão em forma de sigla, e tem colunas ainda com Y/N

In [301]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 45 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id_solicitante                      20000 non-null  int64  
 1   produto_solicitado                  20000 non-null  int64  
 2   dia_vencimento                      20000 non-null  int64  
 3   tipo_endereco                       20000 non-null  int64  
 4   idade                               20000 non-null  int64  
 5   estado_civil                        20000 non-null  int64  
 6   qtde_dependentes                    20000 non-null  int64  
 7   grau_instrucao                      20000 non-null  int64  
 8   nacionalidade                       20000 non-null  int64  
 9   estado_onde_nasceu                  20000 non-null  str    
 10  estado_onde_reside                  20000 non-null  str    
 11  possui_telefone_residencial         20000 non-null  

Mapeando colunas binárias

In [302]:
colunas_binarias = [
    'possui_telefone_residencial', 
    'possui_telefone_celular', 
    'vinculo_formal_com_empresa'
]

for col in colunas_binarias:
    train_df[col] = train_df[col].map({'Y': 1, 'N': 0})
    test_df[col] = test_df[col].map({'Y': 1, 'N': 0})

Corrigindo os espaços em branco

In [303]:
colunas_com_espacos = [
    'codigo_area_telefone_residencial', 
    'codigo_area_telefone_trabalho', 
    'estado_onde_trabalha'
]

# Substitui qualquer campo que tenha apenas espaços (ou vazio) por '-1'
for col in colunas_com_espacos:
    train_df[col] = train_df[col].replace(r'^\s*$', '-1', regex=True)
    test_df[col] = test_df[col].replace(r'^\s*$', '-1', regex=True)

One-Hot Encoding para os Estados

In [304]:
colunas_estados = [
    'estado_onde_nasceu', 
    'estado_onde_reside', 
    'estado_onde_trabalha'
]

# Aplica o get_dummies gerando 1 e 0 (dtype=int) e removendo o primeiro estado (drop_first=True)
train_df = pd.get_dummies(train_df, columns=colunas_estados, drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, columns=colunas_estados, drop_first=True, dtype=int)

colunas_area = ['codigo_area_telefone_residencial', 'codigo_area_telefone_trabalho']
train_df = pd.get_dummies(train_df, columns=colunas_area, drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, columns=colunas_area, drop_first=True, dtype=int)

In [305]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Columns: 276 entries, id_solicitante to codigo_area_telefone_trabalho_97
dtypes: bool(6), float64(11), int64(259)
memory usage: 41.3 MB


# Criar 2 colunas novas para melhorar a acurácia

In [306]:
# Criando a Renda Total
train_df['renda_total'] = train_df['renda_mensal_regular'] + train_df['renda_extra']
test_df['renda_total'] = test_df['renda_mensal_regular'] + test_df['renda_extra']

# Criando a relação de dependentes por renda
train_df['renda_por_dependente'] = train_df['renda_total'] / (train_df['qtde_dependentes'] + 1)
test_df['renda_por_dependente'] = test_df['renda_total'] / (test_df['qtde_dependentes'] + 1)

# Nenhuma coluna string restante, modelo pronto para o treinamento

## Primeiro separar o gabarito

In [307]:

X = train_df.drop(columns=['inadimplente', 'id_solicitante']) 
y = train_df['inadimplente']

Dividindo 80/20 treino e validação

In [308]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho do treino: {X_train.shape[0]} linhas")
print(f"Tamanho da validação: {X_val.shape[0]} linhas")

Tamanho do treino: 16000 linhas
Tamanho da validação: 4000 linhas


## Usando o XGBoost para Treinamento

In [309]:
from xgboost import XGBClassifier

contagem_classes = y_train.value_counts()
peso_balanceamento = contagem_classes[0] / contagem_classes[1]

print(f"Proporção de balanceamento calculada: {peso_balanceamento:.2f}")

modelo_xgb = XGBClassifier(
    random_state=42, 
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=peso_balanceamento,
    eval_metric='logloss',
    n_jobs=-1 # Usa todos os núcleos do processador
)

modelo_xgb.fit(X_train, y_train)

previsoes_xgb = modelo_xgb.predict(X_val)

print("\n--- Relatório de Classificação (XGBoost) ---")
print(classification_report(y_val, previsoes_xgb))
print(f"Acurácia: {accuracy_score(y_val, previsoes_xgb):.4f}")

Proporção de balanceamento calculada: 0.99

--- Relatório de Classificação (XGBoost) ---
              precision    recall  f1-score   support

           0       0.60      0.58      0.59      2027
           1       0.58      0.61      0.60      1973

    accuracy                           0.59      4000
   macro avg       0.59      0.59      0.59      4000
weighted avg       0.59      0.59      0.59      4000

Acurácia: 0.5935


Criando o X test

In [310]:
X_test = test_df.drop(columns=['id_solicitante'])

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## Fazendo as previsões e Criando a Submissão 

In [311]:
previsoes_finais = modelo_xgb.predict(X_test)

submissao = pd.DataFrame({
    'id_solicitante': test_df['id_solicitante'],
    'inadimplente': previsoes_finais 
})

submissao.to_csv('submission.csv', index=False)
print("Arquivo 'submission.csv' gerado com sucesso e pronto para o Kaggle!")

Arquivo 'submission.csv' gerado com sucesso e pronto para o Kaggle!
